In [48]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

## importing dataset


In [33]:
import pandas as pd

file_path = "/Users/meghna/Desktop/sih-2023-it-log-master/ml/financial_transaction_dataset_with_risk.csv"

with open(file_path, "r") as f:
    print(f.readline())  # Print the first line to check delimiter

TransactionID,UserID,SourceAccount,DestinationAccount,TransactionAmount,OldBalanceOrig,NewBalanceOrig,CurrencyType,TransactionType,TransactionStatus,TransactionTimestamp,IPAddress,Geolocation,PaymentMethod,RiskLevel



In [34]:
import pandas as pd

file_path = "/Users/meghna/Desktop/sih-2023-it-log-master/ml/financial_transaction_dataset_with_risk.csv"
df = pd.read_csv(file_path)

print(df.head())

                          TransactionID  UserID SourceAccount  \
0  9bc5d2dd-db2b-42b4-bbc7-62defa36da6a  U41305    AC97946223   
1  b0822d9c-3490-4474-a961-70ba12157595  U70172    AC83686632   
2  c438519b-22b3-4853-ab27-4410933f07c6  U68485    AC71698098   
3  9904e405-df5e-40a9-9f55-e712be8cfd88  U84632    AC51246662   
4  48368571-da30-4375-a6fb-4e4512cac432  U67188    AC51754796   

  DestinationAccount  TransactionAmount  OldBalanceOrig  NewBalanceOrig  \
0         AC46164699           13902.84        30274.79        30274.79   
1         AC16773704            3173.18        18731.22        18731.22   
2         AC80537613           36157.78        39355.26         3197.48   
3         AC34995262           26857.77        75404.58        75404.58   
4         AC17841469           16829.05        91437.74        91437.74   

  CurrencyType TransactionType TransactionStatus TransactionTimestamp  \
0          INR        Purchase           Pending  2025-02-06 20:53:49   
1          G

## dropping redundant columns

In [35]:
# ✅ Step 4: Display Dataset Info
print("Dataset Shape:", df.shape)
print("\nDataset Preview:\n", df.head())

Dataset Shape: (500, 15)

Dataset Preview:
                           TransactionID  UserID SourceAccount  \
0  9bc5d2dd-db2b-42b4-bbc7-62defa36da6a  U41305    AC97946223   
1  b0822d9c-3490-4474-a961-70ba12157595  U70172    AC83686632   
2  c438519b-22b3-4853-ab27-4410933f07c6  U68485    AC71698098   
3  9904e405-df5e-40a9-9f55-e712be8cfd88  U84632    AC51246662   
4  48368571-da30-4375-a6fb-4e4512cac432  U67188    AC51754796   

  DestinationAccount  TransactionAmount  OldBalanceOrig  NewBalanceOrig  \
0         AC46164699           13902.84        30274.79        30274.79   
1         AC16773704            3173.18        18731.22        18731.22   
2         AC80537613           36157.78        39355.26         3197.48   
3         AC34995262           26857.77        75404.58        75404.58   
4         AC17841469           16829.05        91437.74        91437.74   

  CurrencyType TransactionType TransactionStatus TransactionTimestamp  \
0          INR        Purchase           

## label encoding Source and Destination IPs (safe and malicious)

In [49]:
encoders = {}  # Dictionary to store label encoders for each column

for col in ["CurrencyType", "TransactionType", "TransactionStatus", "PaymentMethod"]:
    encoders[col] = LabelEncoder()
    df[col] = encoders[col].fit_transform(df[col])
    
    features = ["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig", "CurrencyType",
            "TransactionType", "TransactionStatus", "PaymentMethod"]
target = "RiskLevel"

X_train, X_test, y_train, y_test = train_test_split(df[features], df[target], test_size=0.2, random_state=42)
## splitting into X and Y
scaler = StandardScaler()
X_train[["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig"]] = scaler.fit_transform(X_train[["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig"]])
X_test[["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig"]] = scaler.transform(X_test[["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig"]])


In [50]:
base_models = [
    ('linear', LinearRegression()),
    ('random_forest', RandomForestRegressor(n_estimators=200, random_state=42)),
    ('xgboost', XGBRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42)),
    ('lightgbm', LGBMRegressor(n_estimators=200, learning_rate=0.05, max_depth=6, random_state=42))
]


## importing catboost regressor

In [39]:
stacking_model = StackingRegressor(estimators=base_models, final_estimator=GradientBoostingRegressor(n_estimators=100, learning_rate=0.1, random_state=42))


## importing lightgbm regressor

In [40]:
print("\nTraining the Stacking Model...")
stacking_model.fit(X_train, y_train)


Training the Stacking Model...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000113 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 422
[LightGBM] [Info] Number of data points in the train set: 400, number of used features: 7
[LightGBM] [Info] Start training from score 0.681425
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No furthe

StackingRegressor(estimators=[('linear', LinearRegression()),
                              ('random_forest',
                               RandomForestRegressor(n_estimators=200,
                                                     random_state=42)),
                              ('xgboost',
                               XGBRegressor(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric=None,...
                                            max_delta_step=None, max_depth=6,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=200, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=42, ...)),
                              ('lightgbm',
                               LGBMRegressor(learning_rate=0.05, max_depth=6,
                                             n_estimators=200,
                                             random_state=42))],
                  final_estimator=GradientBoostingRegressor(random_state=42))

## importing xgboost regressor

In [52]:
y_pred = stacking_model.predict(X_test)

# ✅ Step 11: Evaluate Model Performance
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"\n📊 Model Performance:")
print(f"MSE = {mse:.4f}")
print(f"R² Score = {r2:.4f}")


📊 Model Performance:
MSE = 0.1236
R² Score = -0.4587


## stacking these models together

In [55]:
joblib.dump(stacking_model, "transac_fraud_model.pkl")
joblib.dump(encoders, "transac_encoder.pkl")
joblib.dump(scaler, "transac_scaler.pkl")

['transac_scaler.pkl']

## implementing kfold

In [54]:
# ✅ Define new transaction
new_transaction = pd.DataFrame({
    "TransactionAmount": [30000],
    "OldBalanceOrig": [80000],
    "NewBalanceOrig": [50000],
    "CurrencyType": ["USD"],  # This might be an unseen category
    "TransactionType": ["Transfer"],
    "TransactionStatus": ["Success"],
    "PaymentMethod": ["Credit Card"]
})

# ✅ Encode categorical variables safely
for col in ["CurrencyType", "TransactionType", "TransactionStatus", "PaymentMethod"]:
    if col in encoders:  # Ensure encoder exists
        new_transaction[col] = new_transaction[col].map(lambda x: encoders[col].transform([x])[0] if x in encoders[col].classes_ else -1)

new_transaction[["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig"]] = scaler.transform(new_transaction[["TransactionAmount", "OldBalanceOrig", "NewBalanceOrig"]])

# ✅ Ensure all required columns exist in new_transaction
for col in X_train.columns:
    if col not in new_transaction.columns:
        new_transaction[col] = 0  # Fill missing columns with 0

# ✅ Predict Risk Level
predicted_risk = stacking_model.predict(new_transaction)
print(f"\n🔍 Predicted Risk Level for New Transaction: {predicted_risk[0]:.4f}")



🔍 Predicted Risk Level for New Transaction: 0.8032
